# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as attributes (not as a dict)
meta = dataset.metadata
print(f"Title: {meta.name}\nDate Published: {meta.datePublished}\nVersion: {meta.version}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their fields, and columns, using their `@id`s
# This is important to know the available data tables and their structure before extraction

# Get list of record set metadata objects
record_sets = list(meta.recordSet) if hasattr(meta, 'recordSet') and meta.recordSet else []
if not record_sets:
    print("No record sets found in this dataset metadata. Please check the schema or metadata definition.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}  (name: {rs.get('name', '(no name)')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("  Columns:")
            for col in columns:
                print(f"    - {col['@id']} (name: {col.get('name', '')})")
        print('')

In [ ]:
# As an additional example, show the first record of each available record set (if any)
# Reference record set by @id
for rs in (record_sets if record_sets else []):
    rs_id = rs['@id']
    print(f"\nFirst record for Record Set {rs_id}:")
    try:
        recs = dataset.records(record_set=rs_id)
        first = next(recs)
        print(first)
    except Exception as e:
        print(f"No records could be loaded or error: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract data from all available record sets
# Replace <record_set_ids> with actual @id(s) found above. If none, leave empty.
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}")

# For further operations, pick the first record set if available
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"Main DataFrame columns: {main_df.columns.tolist()}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if main_record_set_id is not None:
    df = main_df.copy()
    print(f"Shape of main record set DataFrame: {df.shape}")
    # Pick a numeric field (try a few common protein fields; fallback to first numeric column)
    possible_numeric = [col for col in df.columns if any(k in col.lower() for k in ['abund', 'intensity', 'mw', 'count', 'coverage', 'number', 'pi'])]
    if not possible_numeric:
        # Try to auto-detect numeric fields
        possible_numeric = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
        print(f"Using '{numeric_field_id}' as numeric field for demonstration.")
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean()  # Use mean as threshold
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (original n={len(df)}, filtered n={len(filtered_df)})")
            display(filtered_df[[numeric_field_id]].head())

            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"First rows with normalized {numeric_field_id}:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try to group by a field if available
            candidate_groups = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 10 and df[col].dtype == object]
            if candidate_groups:
                group_field_id = candidate_groups[0]
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped_df.to_frame())
        except Exception as e:
            print(f"EDA not possible for field {numeric_field_id}: {e}")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No suitable record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id is not None and possible_numeric:
    numeric_field_id = possible_numeric[0]
    plt.figure(figsize=(8,5))
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    main_df[numeric_field_id].plot.hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} in main record set")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Optional: if a grouping field was found, do a boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,5))
        main_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Croissant dataset metadata and explored available record sets and fields via their `@id`.
- Data was extracted for each record set and basic exploratory data analysis was performed on numerical fields.
- Visualizations such as histograms and grouped boxplots help understand distributions of protein abundance and other numeric measurements.
- This workflow can be extended to perform domain-specific analysis and advanced processing for downstream machine learning tasks.